In [9]:
import asyncio
import os
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from agents import Agent, Runner, SQLiteSession
from agents.mcp import MCPServerStreamableHttp
from IPython.display import display, Markdown, JSON, HTML
import json


def run_mcp_in_fresh_loop(coro):
    """Run MCP async code off the Jupyter kernel loop.

    Jupyter's asyncio + AnyIO TaskGroups used by MCP streamable HTTP often
    cancel each other (CancelledError during session.initialize). A dedicated
    thread with its own event loop avoids that.
    """

    def _runner():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(_runner).result()


In [10]:
load_dotenv(override=True)
MODEL = "gpt-5-nano"
_mcp_url = os.getenv("MCP_SERVER")
if not _mcp_url:
    raise RuntimeError("Set MCP_SERVER in .env (MCP server URL).")
PARAMS = {"url": _mcp_url, "timeout": 30}

INSTRUCTIONS = "Play the Alliance game. Use your tools to check status, send messages, and choose who to support."
REGISTER = "Matrix"


In [11]:
# Discover available tools from the MCP server (isolated loop — see run_mcp_in_fresh_loop)
async def _fetch_tools():
    async with MCPServerStreamableHttp(params=PARAMS) as mcp:
        tools = await mcp.list_tools()
        return [
            {
                "name": t.name,
                "description": t.description or "",
                "inputSchema": getattr(t, "inputSchema", None),
            }
            for t in tools
        ]


tool_rows = run_mcp_in_fresh_loop(_fetch_tools())
display(Markdown(f"## Found {len(tool_rows)} tools:\n"))
for row in tool_rows:
    display(Markdown(f"### 🔧 {row['name']}"))
    display(Markdown(f"**Description:** {row['description']}"))
    if row["inputSchema"]:
        display(Markdown("**Parameters:**"))
        display(JSON(row["inputSchema"]))
    display(Markdown("---"))

Task was destroyed but it is pending!
task: <Task pending name='Task-99' coro=<<async_generator_athrow without __name__>()>>
/Users/mubaraq/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/asyncio/base_events.py:750: RuntimeWarning: coroutine method 'aclose' of 'Response.aiter_bytes' was never awaited
  self._ready.clear()


## Found 6 tools:


### 🔧 register

**Description:** Register a new tank to join the game. Call this FIRST before any other action. Your tank name IS your token - reconnect to this MCP endpoint with header "x-player-token: YourTankName" to authenticate all future tool calls. If you lose connection, just register again with the same name to reconnect. Name must be unique, max 20 characters. Registration is only open while the game is in the lobby (waiting for players).

**Parameters:**

...

### 🔧 get_game_state

**Description:** Get the full current game state. Returns: status (lobby/running/ended), currentTurnId (whose turn it is), turnOrder, and all tanks. Each turn has a 60-second time limit - if you don't use both dice in time, your turn is skipped. Use this to check game status and whose turn it is. For tactical decisions (what can I do?), use get_valid_actions instead.

**Parameters:**

<IPython.core.display.JSON object>

---